# Creating a simple map with flight departure delays
**Burhan Zafar**

In this notebook I make interactive Plotly maps of where flights from Seattle (SEA) go and how delayed they are, then save them as HTML so I can put them on my GitHub Pages site.

Data:
* The SEA flight departure delay dataset (2025, six airlines), from the U.S. Bureau of Transportation Statistics
* A lookup table of airport codes, names, and lat/lon locations

| Carrier Code | Airline |
|----------|----------|
| AS | Alaska |
| B6 | JetBlue |
| AA | American Airlines |
| DL | Delta |
| UA | United Airlines |
| WN | Southwest |

In [6]:
# import packages
import pandas as pd
import numpy as np
import plotly.express as px  # new: for quick interactive charts and maps
import plotly.io as pio  # new: for saving charts as html files
import plotly.graph_objects as go  # new: for building the line map by hand
from plotly.colors import sample_colorscale  # turns a number into a color from a color scale

In [7]:
# load the flight departure delay dataset
df = pd.read_csv("/content/SEA_Departure_2025_moreairlines (1).csv")
print(df.shape)
df.head()

(92088, 17)


,Carrier Code,Date (MM/DD/YYYY),Flight Number,Tail Number,Destination Airport,Scheduled departure time,Actual departure time,Scheduled elapsed time (Minutes),Actual elapsed time (Minutes),Departure delay (Minutes),Wheels-off time,Taxi-Out time (Minutes),Delay Carrier (Minutes),Delay Weather (Minutes),Delay National Aviation System (Minutes),Delay Security (Minutes),Delay Late Aircraft Arrival (Minutes)
0,AS,1/1/25,2,N801AK,DCA,13:59,13:56,300.0,275.0,-3.0,14:14,18.0,0.0,0.0,0.0,0.0,0.0
1,AS,1/1/25,4,N568AS,DCA,8:34,8:26,306.0,286.0,-8.0,8:49,23.0,0.0,0.0,0.0,0.0,0.0
2,AS,1/1/25,16,N710AL,JFK,14:11,14:09,318.0,302.0,-2.0,14:27,18.0,0.0,0.0,0.0,0.0,0.0
3,AS,1/1/25,22,N253AK,JFK,23:10,23:06,319.0,285.0,-4.0,23:19,13.0,0.0,0.0,0.0,0.0,0.0
4,AS,1/1/25,26,N514AS,JFK,6:49,6:47,322.0,300.0,-2.0,7:02,15.0,0.0,0.0,0.0,0.0,0.0


The data has about 111,000 flights out of SEA in 2025. Each row is one flight with its airline, destination and departure delay in minutes (negative means it left early).

In [8]:
# average delay for each airline overall, just to get a feel for it
df.groupby("Carrier Code")["Departure delay (Minutes)"].mean().round(1).sort_values()

,Departure delay (Minutes)
Carrier Code,
DL,8.0
UA,9.0
AS,10.2
AA,15.3
WN,15.5
B6,37.4


Delta is the most on time (7.5 min average) and JetBlue is the worst by far (37.4 min), but JetBlue only has 560 flights so that number is less reliable. Alaska, the biggest airline at SEA, averages about 10 minutes.

## Making the map
Plan:
1. Group by airline and destination and average the delay.
2. Join with the lat/lon lookup table.
3. Plot the points on a map: color = delay, size = number of flights.

In [9]:
# group by airline and destination, and get the mean delay, std, and number of flights
avg_delay_by_carrier_destination = (
    df.groupby(["Carrier Code", "Destination Airport"], as_index=False)
      .agg(
          mean_departure_delay=("Departure delay (Minutes)", "mean"),  # average delay
          std_departure_delay=("Departure delay (Minutes)", "std"),  # how spread out the delays are
          flight_count=("Departure delay (Minutes)", "count")  # number of flights
      )
      .sort_values("mean_departure_delay", ascending=False)
      .dropna()  # drop routes with only one flight (std is NaN for those)
)
avg_delay_by_carrier_destination

,Carrier Code,Destination Airport,mean_departure_delay,std_departure_delay,flight_count
82,B6,BOS,43.845144,154.088678,381
97,DL,HNL,28.347328,123.089934,262
132,WN,DAL,26.406250,28.165830,128
8,AA,PHX,24.625000,69.825012,32
83,B6,JFK,23.793296,67.024117,179
...,...,...,...,...,...
114,DL,RDU,1.969388,25.452383,98
113,DL,PSP,1.846939,33.048029,98
107,DL,MSP,1.487500,21.345701,400
86,DL,AUS,0.292994,23.212027,157


In [10]:
# load the airport lookup table (code, name, lat, lon)
locations = pd.read_csv("airport_lookup_table.csv")
locations

,destination,airport_name,latitude,longitude
0,ABQ,Albuquerque International Sunport,35.0402,-106.6090
1,ANC,Ted Stevens Anchorage International Airport,61.1743,-149.9985
2,ATL,Hartsfield-Jackson Atlanta International Airport,33.6407,-84.4277
3,AUS,Austin-Bergstrom International Airport,30.1975,-97.6664
4,BNA,Nashville International Airport,36.1245,-86.6782
...,...,...,...,...
72,SNA,John Wayne Airport,33.6757,-117.8678
73,STL,St. Louis Lambert International Airport,38.7487,-90.3700
74,STS,Charles M. Schulz-Sonoma County Airport,38.5089,-122.8129
75,TPA,Tampa International Airport,27.9755,-82.5332


In [11]:
# check: are there any destinations in the flight data that the lookup table is missing?
missing = set(df["Destination Airport"]) - set(locations["destination"])
print("missing airports:", missing)

missing airports: set()


No airports are missing, so the inner join won't silently drop any routes. The lookup table was made with ChatGPT though, and the coordinates for the big ones (ANC, LAX, PHX, HNL) look right. Still, since it's AI-made, I wouldn't fully trust every row without checking. A wrong lat/lon would put a dot in the wrong place and nobody would notice.

In [12]:
# join the delay data with the airport locations using the 3-letter code
df_merged = pd.merge(
    avg_delay_by_carrier_destination,
    locations,
    left_on="Destination Airport",
    right_on="destination",
    how="inner"
)
df_merged

,Carrier Code,Destination Airport,mean_departure_delay,std_departure_delay,flight_count,destination,airport_name,latitude,longitude
0,B6,BOS,43.845144,154.088678,381,BOS,Boston Logan International Airport,42.3656,-71.0096
1,DL,HNL,28.347328,123.089934,262,HNL,Daniel K. Inouye International Airport,21.3187,-157.9225
2,WN,DAL,26.406250,28.165830,128,DAL,Dallas Love Field,32.8471,-96.8517
3,AA,PHX,24.625000,69.825012,32,PHX,Phoenix Sky Harbor International Airport,33.4342,-112.0116
4,B6,JFK,23.793296,67.024117,179,JFK,John F. Kennedy International Airport,40.6413,-73.7781
...,...,...,...,...,...,...,...,...,...
133,DL,RDU,1.969388,25.452383,98,RDU,Raleigh-Durham International Airport,35.8801,-78.7880
134,DL,PSP,1.846939,33.048029,98,PSP,Palm Springs International Airport,33.8297,-116.5070
135,DL,MSP,1.487500,21.345701,400,MSP,Minneapolis-Saint Paul International Airport,44.8848,-93.2223
136,DL,AUS,0.292994,23.212027,157,AUS,Austin-Bergstrom International Airport,30.1975,-97.6664


For now I'll just look at Alaska, since it has the most flights.

In [13]:
# keep only alaska flights
df_alaska = df_merged[df_merged["Carrier Code"] == "AS"]
print(len(df_alaska), "alaska destinations")
df_alaska.head()

73 alaska destinations


,Carrier Code,Destination Airport,mean_departure_delay,std_departure_delay,flight_count,destination,airport_name,latitude,longitude
13,AS,HNL,17.304300,29.392262,907,HNL,Daniel K. Inouye International Airport,21.3187,-157.9225
14,AS,SIT,17.008368,31.545884,239,SIT,Sitka Rocky Gutierrez Airport,57.0471,-135.3616
19,AS,EWR,16.101499,44.033384,867,EWR,Newark Liberty International Airport,40.6895,-74.1745
22,AS,SFO,15.173650,32.927963,2315,SFO,San Francisco International Airport,37.6213,-122.3790
24,AS,DEN,14.876320,36.493263,1326,DEN,Denver International Airport,39.8561,-104.6737


In [14]:
# first try at the map: size = number of flights, color = average delay
fig = px.scatter_map(df_alaska, lat="latitude", lon="longitude",
                     size="flight_count",
                     color="mean_departure_delay",
                     color_continuous_scale=px.colors.cyclical.IceFire)
fig.show()

It works, but the default zoom is weird and the hover shows a bunch of extra stuff. Fixes:
* set the center and zoom so the whole US shows
* make the hover only show airport name, number of flights and average delay
* save it as an HTML file for my website

In [15]:
# cleaned up map
fig = px.scatter_map(df_alaska, lat="latitude", lon="longitude",
                     size="flight_count",
                     color="mean_departure_delay",
                     color_continuous_scale=px.colors.cyclical.IceFire,
                     zoom=3, center={"lat": 39, "lon": -98},
                     hover_name="airport_name",  # airport name as the big title in the hover
                     hover_data={
                         "flight_count": True,  # show number of flights
                         "mean_departure_delay": ":.1f",  # show delay with 1 decimal
                         "latitude": False, "longitude": False  # hide lat/lon
                     },
                     labels={"flight_count": "Flights", "mean_departure_delay": "Avg delay (min)"},
                     title="Alaska Airlines flights from SEA: average departure delay by destination (2025)")
fig.show()

# save the map as an html file to embed on my github pages site
pio.write_html(fig, file="Alaska_destinations_delays.html", auto_open=False, include_plotlyjs="cdn")

**What the map shows:** Alaska's biggest routes from SEA are Anchorage (4,407 flights), LA, San Diego and Phoenix. The most on-time destinations are John Wayne/Orange County (3.2 min average delay), Milwaukee and Palm Springs. The worst are Honolulu (17.3 min), Sitka, Newark and SFO (15.2 min). Long flights to Hawaii and the East Coast seem to leave later on average, and SFO is a famously delay-heavy airport, so that one makes sense.

## Embedding on my website
1. Save the map with `pio.write_html()` (done above)
2. Download `Alaska_destinations_delays.html` from the Colab files panel
3. Add it to my website's GitHub repo
4. On the page, add:

```
<iframe
  src="Alaska_destinations_delays.html"
  width="100%"
  height="600"
  frameborder="0">
</iframe>
```
5. Add a few sentences around the map explaining what it shows (like the paragraph above).

## Challenge: flight path lines

Instead of dots, draw a line from SEA to each destination. I used the Plotly US flight paths example: each line is its own `Scattergeo` trace with two points, start (SEA) and end (the destination).

To show the delay, I made each line's **color** and **thickness** depend on the average delay. Plotly can't do a color scale on separate line traces automatically, so I turn each delay into a 0–1 number and use `sample_colorscale` to pick the color myself.

In [16]:
# seattle's coordinates (where every line starts)
SEA_LAT = 47.443546
SEA_LON = -122.301659

# turn each route's delay into a number from 0 (least delayed) to 1 (most delayed)
delays = df_alaska["mean_departure_delay"]
delay_scaled = (delays - delays.min()) / (delays.max() - delays.min())

# pick a color for each line from a yellow -> red scale
line_colors = sample_colorscale("YlOrRd", list(0.15 + 0.85 * delay_scaled))

fig = go.Figure()

# one line per destination, from SEA to that airport
for i in range(len(df_alaska)):
    row = df_alaska.iloc[i]
    fig.add_trace(
        go.Scattergeo(
            lon=[SEA_LON, row["longitude"]],
            lat=[SEA_LAT, row["latitude"]],
            mode="lines",
            line=dict(width=1 + 5 * delay_scaled.iloc[i],  # thicker = more delayed
                      color=line_colors[i]),  # redder = more delayed
            hoverinfo="text",
            text=f"{row['airport_name']}<br>Flights: {row['flight_count']}<br>Avg delay: {row['mean_departure_delay']:.1f} min"
        )
    )

# dots for the airports on top of the lines
fig.add_trace(
    go.Scattergeo(
        lon=df_alaska["longitude"], lat=df_alaska["latitude"],
        mode="markers", hoverinfo="text", text=df_alaska["airport_name"],
        marker=dict(size=4, color="black")
    )
)

# a fake invisible trace just so a color bar legend shows up
fig.add_trace(
    go.Scattergeo(
        lon=[None], lat=[None], mode="markers",
        marker=dict(colorscale="YlOrRd", cmin=delays.min(), cmax=delays.max(), color=[delays.min()],
                    showscale=True, colorbar=dict(title="Avg delay (min)")),
        hoverinfo="skip"
    )
)

fig.update_layout(
    title_text="Alaska Airlines flight paths from SEA (thicker and redder = more delayed)",
    showlegend=False,
    geo=dict(
        scope="north america",
        projection_type="azimuthal equal area",
        center=dict(lat=45, lon=-120),
        projection_scale=1.6,
        showland=True,
        landcolor="rgb(243, 243, 243)",
        countrycolor="rgb(204, 204, 204)",
    ),
)
fig.show()

# save this one too for the website
pio.write_html(fig, file="Alaska_flight_paths_delays.html", auto_open=False, include_plotlyjs="cdn")

The line map makes the pattern easier to see than the dots. The thick red lines go to Hawaii, Newark, Sitka and SFO, and the thin yellow ones are mostly short West Coast and Mountain routes. One downside: long lines (like to Hawaii or the East Coast) look "heavier" just because they're longer, even if the delay isn't that different, so the dot map is probably more honest for comparing delays. The line map is better for showing *where* Alaska flies.


Categorizing the following by states:

---



---


In [18]:
import pandas as pd

# Mapping of common airport codes in the dataset to their US States
# This covers the destinations found in the locations and flight datasets
airport_state_map = {
    'ABQ': 'NM', 'ANC': 'AK', 'ATL': 'GA', 'AUS': 'TX', 'BNA': 'TN',
    'BOS': 'MA', 'BUR': 'CA', 'BWI': 'MD', 'CLE': 'OH', 'CLT': 'NC',
    'CMH': 'OH', 'CVG': 'KY', 'DAL': 'TX', 'DCA': 'VA', 'DEN': 'CO',
    'DFW': 'TX', 'DTW': 'MI', 'EWR': 'NJ', 'FAI': 'AK', 'FAT': 'CA',
    'FLL': 'FL', 'GEG': 'WA', 'HDN': 'CO', 'HNL': 'HI', 'IAD': 'VA',
    'IAH': 'TX', 'IND': 'IN', 'JFK': 'NY', 'KOA': 'HI', 'LAS': 'NV',
    'LAX': 'CA', 'LGB': 'CA', 'LIH': 'HI', 'MCI': 'MO', 'MCO': 'FL',
    'MIA': 'FL', 'MKE': 'WI', 'MSP': 'MN', 'MSY': 'LA', 'OAK': 'CA',
    'OGG': 'HI', 'OKC': 'OK', 'OMA': 'NE', 'ONT': 'CA', 'ORD': 'IL',
    'PDX': 'OR', 'PHL': 'PA', 'PHX': 'AZ', 'PIT': 'PA', 'PSC': 'WA',
    'PSP': 'CA', 'PVD': 'RI', 'RDM': 'OR', 'RDU': 'NC', 'RNO': 'NV',
    'ROC': 'NY', 'SAN': 'CA', 'SAT': 'TX', 'SFO': 'CA', 'SJC': 'CA',
    'SLC': 'UT', 'SMF': 'CA', 'SNA': 'CA', 'STL': 'MO', 'STS': 'CA',
    'TPA': 'FL', 'TUS': 'AZ', 'SIT': 'AK', 'KTN': 'AK', 'JNU': 'AK',
    'PSG': 'AK', 'WRG': 'AK', 'YAK': 'AK', 'CDV': 'AK', 'OME': 'AK',
    'OTZ': 'AK', 'SCC': 'AK', 'BLI': 'WA', 'GEG': 'WA', 'ALW': 'WA',
    'EAT': 'WA', 'YKM': 'WA', 'PUW': 'WA', 'LWS': 'ID', 'BOI': 'ID',
    'SUN': 'ID', 'BZN': 'MT', 'MSO': 'MT', 'FCA': 'MT', 'GTF': 'MT',
    'HLN': 'MT', 'BTM': 'MT', 'BIL': 'MT', 'RKS': 'WY', 'JAC': 'WY'
}

# Map the states to the merged dataframe using the Destination Airport code
df_merged['State'] = df_merged['Destination Airport'].map(airport_state_map)

# Group and categorize by State to see average delays and flight counts
state_summary = df_merged.groupby('State').agg(
    avg_delay=('mean_departure_delay', 'mean'),
    total_flights=('flight_count', 'sum'),
    unique_airports=('Destination Airport', 'nunique')
).round(1).sort_values(by='avg_delay', ascending=False)

print("Summary of Departure Delays Categorized by Destination State:")
print(state_summary)

Summary of Departure Delays Categorized by Destination State:
       avg_delay  total_flights  unique_airports
State                                           
MA          21.4           1596                1
NJ          17.6           1657                1
HI          16.3           3869                4
NY          15.7           1604                1
IL          14.4           4078                1
CO          13.1           4333                1
ID          13.1            607                1
IN          12.5            463                1
WA          12.2           1759                2
TX          12.0           7609                5
GA          12.0           1240                1
PA          11.9           1330                2
MT          11.9            749                2
MO          11.6           1535                2
NM          11.4            479                1
NV          11.2           4136                2
AZ          10.5           4282                2
AK     

In [19]:
import plotly.express as px

# Reset the index of the summary to make 'State' a regular column for plotting
state_plot_df = state_summary.reset_index()

# Create an interactive, sorted horizontal bar chart
fig_state = px.bar(
    state_plot_df,
    x='avg_delay',
    y='State',
    orientation='h',
    color='avg_delay',
    color_continuous_scale='Reds',
    title='Average Flight Departure Delay by Destination State (from SEA, 2025)',
    labels={
        'avg_delay': 'Average Departure Delay (Minutes)',
        'State': 'Destination State'
    },
    hover_data={
        'avg_delay': ':.1f',
        'total_flights': ':,',
        'unique_airports': True
    }
)

# Update layout for cleaner aesthetics, sorted order, and explicit height to fit all states comfortably
fig_state.update_layout(
    yaxis={'categoryorder': 'total ascending'},
    height=800,
    coloraxis_showscale=True,
    coloraxis_colorbar=dict(title="Delay (min)"),
    margin=dict(l=100, r=50, t=80, b=50)
)

fig_state.show()

# Save the state-level visualization to HTML
pio.write_html(fig_state, file="State_destinations_delays.html", auto_open=False, include_plotlyjs="cdn")

In [20]:
def assign_region(state):
    west = ['CA', 'OR', 'WA', 'NV', 'AZ', 'UT', 'ID', 'MT', 'WY', 'CO', 'NM']
    midwest = ['IL', 'OH', 'IN', 'MI', 'WI', 'MN', 'NE', 'MO', 'OK']
    south = ['TX', 'GA', 'TN', 'NC', 'LA', 'FL', 'MD', 'VA']
    east = ['MA', 'NJ', 'NY', 'PA', 'RI']
    non_contiguous = ['AK', 'HI']

    if state in west: return 'West'
    elif state in midwest: return 'Midwest'
    elif state in south: return 'South'
    elif state in east: return 'East'
    elif state in non_contiguous: return 'Non-Contiguous'
    else: return 'Other'

# 1. Create Regional and Traffic Volume Subcategories
state_plot_df['Region'] = state_plot_df['State'].apply(assign_region)

# Categorize traffic volume based on percentiles
q_low, q_high = state_plot_df['total_flights'].quantile([0.33, 0.66])
def assign_volume(flights):
    if flights <= q_low: return 'Low Volume (<600 flights)'
    elif flights <= q_high: return 'Medium Volume (600-1.7k flights)'
    else: return 'High Volume (>1.7k flights)'

state_plot_df['Traffic_Volume'] = state_plot_df['total_flights'].apply(assign_volume)

# 2. Regional Analysis Summary
region_summary = state_plot_df.groupby('Region').agg(
    avg_delay=('avg_delay', 'mean'),
    total_flights=('total_flights', 'sum'),
    number_of_states=('State', 'count')
).round(1).sort_values(by='avg_delay', ascending=False)

# 3. Traffic Volume Analysis Summary
volume_summary = state_plot_df.groupby('Traffic_Volume').agg(
    avg_delay=('avg_delay', 'mean'),
    total_flights=('total_flights', 'sum'),
    number_of_states=('State', 'count')
).round(1).sort_values(by='avg_delay', ascending=False)

print("--- 1. DELAYS BY GEOGRAPHIC REGION ---")
display(region_summary)

print("\n--- 2. DELAYS BY TRAFFIC VOLUME OF DESTINATION STATE ---")
display(volume_summary)

--- 1. DELAYS BY GEOGRAPHIC REGION ---


,avg_delay,total_flights,number_of_states
Region,,,
East,16.6,6187,4
Non-Contiguous,13.4,12917,2
West,10.8,42654,10
South,9.4,17771,8
Midwest,8.2,10096,9
Other,8.2,449,1



--- 2. DELAYS BY TRAFFIC VOLUME OF DESTINATION STATE ---


,avg_delay,total_flights,number_of_states
Traffic_Volume,,,
High Volume (>1.7k flights),12.0,69141,12
Medium Volume (600-1.7k flights),10.9,15104,11
Low Volume (<600 flights),8.7,5829,11


In [26]:
# Visualizing the Subcategories using a Sunburst plot with a discrete qualitative scale
# so that each sector and category path is assigned a distinct, readable color
fig_sunburst = px.sunburst(
    state_plot_df,
    path=['Region', 'Traffic_Volume', 'State'],
    values='total_flights',
    color='Region',
    color_discrete_sequence=px.colors.qualitative.Prism,
    title='Drill-Down: Flights categorized by Region -> Traffic Volume -> State (Color-coded by Region)',
    labels={'avg_delay': 'Avg Delay (min)', 'total_flights': 'Total Flights'}
)

fig_sunburst.update_layout(
    height=700,
    paper_bgcolor='white',
    font=dict(color='black')
)
fig_sunburst.show()

# Save the subcategorized sunburst map to HTML
pio.write_html(fig_sunburst, file="Subcategorized_destinations_delays.html", auto_open=False, include_plotlyjs="cdn")

In [29]:
import pandas as pd
import plotly.express as px
import plotly.io as pio

# Ensure the date column is parsed to extract the month with explicit format to avoid warnings
df['Date_Parsed'] = pd.to_datetime(df['Date (MM/DD/YYYY)'], format='%m/%d/%y', errors='coerce')
df['Month'] = df['Date_Parsed'].dt.month

# Group by Month and Carrier Code to compute average departure delay
heatmap_data = (
    df.groupby(['Month', 'Carrier Code'], as_index=False)['Departure delay (Minutes)']
    .mean()
    .rename(columns={'Departure delay (Minutes)': 'avg_delay'})
)

# Map month numbers to month names for better readability
month_map = {1: 'Jan', 2: 'Feb', 3: 'Mar', 4: 'Apr', 5: 'May', 6: 'Jun',
             7: 'Jul', 8: 'Aug', 9: 'Sep', 10: 'Oct', 11: 'Nov', 12: 'Dec'}
heatmap_data['Month_Name'] = heatmap_data['Month'].map(month_map)

# Pivot the data to create a matrix (index=Carrier Code, columns=Month)
heatmap_pivot = heatmap_data.pivot(index='Carrier Code', columns='Month_Name', values='avg_delay')

# Sort months chronologically in the columns (filtering only months present in our dataset)
present_months = [month_map[m] for m in sorted(heatmap_data['Month'].unique()) if m in month_map]
heatmap_pivot = heatmap_pivot[present_months]

# Generate the interactive Heatmap
fig_heatmap = px.imshow(
    heatmap_pivot,
    labels=dict(x="Month", y="Airline Carrier", color="Avg Delay (min)"),
    x=heatmap_pivot.columns,
    y=heatmap_pivot.index,
    color_continuous_scale='Reds',
    title='SEA Airport (2025): Average Departure Delay (Minutes) by Month and Carrier',
    text_auto='.1f'  # Display values inside cells with one decimal place
)

# Ensure explicit formatting and readability
fig_heatmap.update_layout(
    height=500,
    xaxis_title="Month",
    yaxis_title="Carrier Code",
    plot_bgcolor='white',
    paper_bgcolor='white',
    font=dict(color='black')
)

fig_heatmap.show()

# Save the heatmap to HTML
pio.write_html(fig_heatmap, file="Carrier_Month_Delay_Heatmap.html", auto_open=False, include_plotlyjs="cdn")

In [27]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px

# Create a structured layout with subplots: one for Regions and one for Traffic Volumes
fig_dashboard = make_subplots(
    rows=1, cols=2,
    subplot_titles=("Avg Delay by Geographic Region", "Avg Delay by Destination Traffic Volume"),
    horizontal_spacing=0.15
)

# Sort regional and volume summaries for clean presentation
region_clean = region_summary.reset_index().sort_values('avg_delay', ascending=True)
volume_clean = volume_summary.reset_index().sort_values('avg_delay', ascending=True)

# Use distinct qualitative color schemes so each individual bar has a unique, beautiful color
region_colors = px.colors.qualitative.Pastel
volume_colors = px.colors.qualitative.Safe

# 1. Left Plot: Regions (Using distinct qualitative colors for each region)
fig_dashboard.add_trace(
    go.Bar(
        x=region_clean['avg_delay'],
        y=region_clean['Region'],
        orientation='h',
        marker=dict(
            color=region_colors[:len(region_clean)],
            line=dict(color='black', width=1)
        ),
        name='Regions',
        hovertemplate="<b>%{y}</b><br>Avg Delay: %{x:.1f} min<br><extra></extra>"
    ),
    row=1, col=1
)

# 2. Right Plot: Traffic Volume Categories (Using distinct qualitative colors for each volume tier)
fig_dashboard.add_trace(
    go.Bar(
        x=volume_clean['avg_delay'],
        y=volume_clean['Traffic_Volume'],
        orientation='h',
        marker=dict(
            color=volume_colors[:len(volume_clean)],
            line=dict(color='black', width=1)
        ),
        name='Traffic Volume',
        hovertemplate="<b>%{y}</b><br>Avg Delay: %{x:.1f} min<br><extra></extra>"
    ),
    row=1, col=2
)

# Style the subplots dashboard with high-visibility layouts
fig_dashboard.update_layout(
    title="Interactive Subcategory Dashboard: Distinct Regional & Volume Categorizations",
    title_font=dict(size=16, color='black', family="Arial"),
    showlegend=False,
    height=500,
    margin=dict(t=80, b=50, l=120, r=50),
    plot_bgcolor='white',
    paper_bgcolor='white',
    font=dict(color='black')
)

# Add visible gridlines
fig_dashboard.update_xaxes(title_text="Avg Delay (mins)", title_font=dict(color='black'), gridcolor='LightGray', showgrid=True, linecolor='black')
fig_dashboard.update_yaxes(gridcolor='LightGray', showgrid=False, linecolor='black')

fig_dashboard.show()

# Save the highly visible dashboard to HTML
pio.write_html(fig_dashboard, file="Interactive_Subcategory_Dashboard.html", auto_open=False, include_plotlyjs="cdn")

In [25]:
import plotly.express as px
import plotly.io as pio

# Create a detailed treemap that lets the user drill-down interactively through subcategories
# Using the 'Region' column to assign distinct, visually pleasing discrete colors
fig_treemap = px.treemap(
    state_plot_df,
    path=[px.Constant("United States"), 'Region', 'Traffic_Volume', 'State'],
    values='total_flights',
    color='Region',
    color_discrete_sequence=px.colors.qualitative.Safe,  # Highly accessible, distinct, and visually pleasing palette
    title='Interactive State Delay Treemap (Color-coded by Region, Click to Zoom)',
    labels={'avg_delay': 'Avg Delay (min)', 'total_flights': 'Total Flights'},
    hover_data={
        'avg_delay': ':.1f',
        'total_flights': ':,'
    }
)

# Ensure clean visibility with solid text contrast and white backgrounds
fig_treemap.update_layout(
    height=650,
    margin=dict(t=50, l=10, r=10, b=10),
    paper_bgcolor='white',
    font=dict(color='black')
)

fig_treemap.show()

# Save the interactive treemap as well
pio.write_html(fig_treemap, file="Interactive_Treemap_Delays.html", auto_open=False, include_plotlyjs="cdn")

In [30]:
import pandas as pd

# Group the dataset by carrier to calculate average delay and total flight volume
carrier_stats = df.groupby('Carrier Code').agg(
    avg_delay=('Departure delay (Minutes)', 'mean'),
    flight_volume=('Departure delay (Minutes)', 'count')
).reset_index()

# Calculate the Pearson correlation coefficient
correlation = carrier_stats['avg_delay'].corr(carrier_stats['flight_volume'])

print("Carrier Delay vs. Flight Volume Metrics:")
display(carrier_stats.sort_values(by='flight_volume', ascending=False))
print(f"\nPearson correlation coefficient between carrier average delay and flight volume: {correlation:.4f}")

Carrier Delay vs. Flight Volume Metrics:


,Carrier Code,avg_delay,flight_volume
1,AS,10.185879,63762
4,UA,8.952624,7852
3,DL,7.968899,7781
5,WN,15.498355,6991
0,AA,15.312196,5141
2,B6,37.435714,560



Pearson correlation coefficient between carrier average delay and flight volume: -0.3598


In [31]:
import numpy as np
import plotly.graph_objects as go
from sklearn.linear_model import LinearRegression

# 1. Prepare data from our carrier statistics
X = carrier_stats[['flight_volume']].values
y = carrier_stats['avg_delay'].values

# 2. Fit the linear regression model
model = LinearRegression()
model.fit(X, y)

# 3. Calculate model metrics
slope = model.coef_[0]
intercept = model.intercept_
r_squared = model.score(X, y)
y_pred = model.predict(X)

print("=== Linear Regression Analysis Results ===")
print(f"Equation: Avg Delay = ({slope:.6f} * Flight Volume) + {intercept:.4f}")
print(f"R-squared (Coefficient of Determination): {r_squared:.4f}")

# 4. Create an interactive scatter plot with the regression line
fig_reg = go.Figure()

# Add actual carrier data points
fig_reg.add_trace(go.Scatter(
    x=carrier_stats['flight_volume'],
    y=carrier_stats['avg_delay'],
    mode='markers+text',
    text=carrier_stats['Carrier Code'],
    textposition="top center",
    marker=dict(size=12, color='DarkBlue'),
    name='Actual Carriers'
))

# Create a smooth line for the regression prediction
x_range = np.linspace(0, carrier_stats['flight_volume'].max() * 1.1, 100)
y_line = model.predict(x_range.reshape(-1, 1))

fig_reg.add_trace(go.Scatter(
    x=x_range,
    y=y_line,
    mode='lines',
    line=dict(color='Red', width=2, dash='dash'),
    name=f'Regression Line (R² = {r_squared:.2f})'
))

fig_reg.update_layout(
    title="Linear Regression: Carrier Avg Delay vs. Flight Volume",
    xaxis_title="Flight Volume (Number of Flights)",
    yaxis_title="Average Departure Delay (Minutes)",
    plot_bgcolor='white',
    paper_bgcolor='white',
    font=dict(color='black'),
    showlegend=True
)

fig_reg.update_xaxes(showgrid=True, gridcolor='LightGray')
fig_reg.update_yaxes(showgrid=True, gridcolor='LightGray')

fig_reg.show()

# Save the regression plot to HTML
pio.write_html(fig_reg, file="Carrier_Delay_Regression.html", auto_open=False, include_plotlyjs="cdn")

=== Linear Regression Analysis Results ===
Equation: Avg Delay = (-0.000166 * Flight Volume) + 18.4430
R-squared (Coefficient of Determination): 0.1295


## Resources
I used the class template and the Plotly "lines on maps" example (https://plotly.com/python/lines-on-maps/).